# GRS-34806 MGI Project
## 1) Download the data
This template downloads the UCM data (both mono and multi-labels) in your local Colab environment.

Write the notebook in such a way that it fully runs from start to end without further intervention  
(i.e. do not change the directory structure manually in the mean time).

Good luck!

In [1]:
import os
import zipfile

! git clone https://git.wur.nl/lobry001/ucmdata.git
os.chdir('ucmdata')

with zipfile.ZipFile('UCMerced_LandUse.zip', 'r') as zip_ref:
    zip_ref.extractall('UCMImages')

!mv UCMImages/UCMerced_LandUse/Images .
!rm -rf UCMImages README.md  UCMerced_LandUse.zip
!ls

UCM_images_path = "Images/"
Multilabels_path = "LandUse_Multilabeled.txt"

Cloning into 'ucmdata'...
remote: Enumerating objects: 8, done.
remote: Total 8 (delta 0), reused 0 (delta 0), pack-reused 8 (from 1)
Receiving objects: 100% (8/8), 316.99 MiB | 18.52 MiB/s, done.
Images	LandUse_Multilabeled.txt


In [2]:
## add Weights & Biases for logging
%pip install -q wandb
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: alexander-klupsch (alexander-klupsch-trier-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [8]:
## Setting up our images
import os
dir = "/content/ucmdata/Images"
image_dict = {}

for class_name in os.listdir(dir):
  class_path = os.path.join(dir, class_name)

  if os.path.isdir(class_path):
    image_paths = [os.path.join(class_path, img) for img in os.listdir(class_path) if img.endswith(('.tif'))]
    image_dict[class_name] = image_paths

In [10]:
import random

def create_datasets(image_dict, val_fraction):
  train_dataset = []
  val_dataset = []

  for label, paths in image_dict.items():
    random.shuffle(paths)
    split_idx = int(len(paths) * (1 - val_fraction))
    train_paths = paths[:split_idx]
    val_paths = paths[split_idx:]
    train_dataset.extend([ (path, label) for path in train_paths ])
    val_dataset.extend([ (path, label) for path in val_paths])

    return train_dataset, val_dataset

In [11]:
train_data, val_data = create_datasets(image_dict, 0.2)

# 1. First Approach: Pretrained ResNet18 and finetune

The following code is an implementation of the medium post here:
https://medium.com/@imabhi1216/fine-tuning-a-pre-trained-resnet-18-model-for-image-classification-on-custom-dataset-with-pytorch-02df12e83c2c

It uses a pretrained ResNet18 and finetunes the head for our specific case

In [6]:
import torch
import torchvision.models as models

# Load the pre-trained ResNet-18 model
model = models.resnet18(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 81.3MB/s]


In [7]:
# Modify the last layer of the model
num_classes = 10 # replace with the number of classes in your dataset
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)